# 03_finetune

현재 기본 설정은 `CAMPAIGN="finetune_wd_zero_control"`, `EXECUTE=False`입니다. `02_pretrain.ipynb`를 재실행하지 않고 이미 완료된 `weight_decay=0.0` Parent(`640ff9bf64d22462464d65cb42acb5cc86f3549c88ec568158169015d54a7213`)를 재사용하며, SurvFace 5,319명 전체(`full_fr_resize_only`) MagFace 1개 모델에 대해 fine-tune 단계의 Adam `training.weight_decay`만 `0.0001 → 0.0`으로 바꾸는 단일 변수 대조 항목입니다. 학습 실행 시 `EXECUTE=True`로 변경하세요.

캠페인 경로는 연구 목적에 따라 분리되어 있습니다.
- `CAMPAIGN="finetune_wd_zero_control"` (`purpose="diagnostic_control"`): 완료된 `WD=0` Parent를 재사용하고 SurvFace 5,319명 MagFace 1개의 fine-tune `training.weight_decay=0.0`만 대조합니다. 비교 기준은 동일 Parent로 학습된 기존 `ft_wd=0.0001` 모델(`magface-ad3b19a576ec14a0883c`)이며, `save_freq=1`·`KEEP_RAW_RESULTS=True`로 1·2·5·10·40 epoch 체크포인트를 별도 경로(`finetune_wd_zero_v1`)에 보존합니다.
- `CAMPAIGN="parent_control"` (`purpose="diagnostic_control"`): `WD=0` Parent + 기존 fine-tune `weight_decay=0.0001` 대조 경로(`followup/selection.json`, 40 epoch 완료)입니다.
- `CAMPAIGN="ucface_review"` (`EXECUTE=False`): 현재 완료 모델 상태, UCFace 재현 조건 차이, parent vs fine-tuned Origin 비교를 점검합니다.
- `CAMPAIGN="paper_baseline"`: UCFace 논문 기준 SurvFace 전체 학습 분할(`5,319명 / 220,888장`, `calibration_unseen_in_fr=False`)로 SurvFace를 학습하고, 완료 검증된 SCFace(`two_eye_alignment`) 4개·TinyFace(`pfld_five_point_alignment`) 4개 모델은 재사용합니다.
- `CAMPAIGN="selected"`: FR 학습에서 보정 인물을 제외한 기존 연구 조건(`SurvFace 4,255명 / 176,205장`, `calibration_unseen_in_fr=True`) 경로입니다. 12개 모델이 모두 40 epoch 완료되어 `reuse_completed` 상태입니다.


## 1. 설정
00에서 저장한 HEADS·SEEDS·데이터셋·backbone 설정을 이어받습니다. 새 캠페인은 00 설정 저장 → 01 부모 입력 준비 → 02 부모 학습 완료 후 진행합니다. 검증된 동일 조건의 완료 부모만 재사용할 수 있습니다. 새 커널에서 공통 부모를 독립 복사해 각 head와 backbone을 함께 학습합니다. SETTINGS_PATH는 00과 같아야 합니다.

선택지·범위가 적힌 변수만 필요한 조건으로 조정합니다. 공통 변경은 00과 같은 SETTINGS_PATH를 사용하는 후속 단계에 일관되게 반영합니다.


In [1]:
# cell 1 : 사용자 설정 및 캠페인 선택
# 목적: 완료 부모로 학습할 데이터셋·head·seed와 전처리·학습 실행 조건을 선택합니다.

from pathlib import Path
import sys, json, os
BASE_DIR = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "fine-tune/modules").is_dir())  # 저장소 위치 자동 탐색값입니다. 직접 변경하지 않습니다.
DATASET_ROOT = Path(os.environ.get("FACE_DATASET_ROOT", "F:/face_datasets" if os.name == "nt" else "/data/face_datasets"))  # 공유 데이터 루트; 환경변수 FACE_DATASET_ROOT 또는 이 Path를 실제 경로로 지정합니다.
MODEL_ROOT = Path(os.environ.get("FACE_MODEL_ROOT", "F:/models" if os.name == "nt" else "/data/models"))  # 공유 모델 루트; 환경변수 FACE_MODEL_ROOT 또는 이 Path를 실제 경로로 지정합니다.
SETTINGS_PATH = DATASET_ROOT / ".state/facex/shared_v1_boundary/notebook_settings_mobilefacenet.json"  # 00에서 저장하고 01~04에서 함께 읽을 JSON입니다. backbone/실험별로 파일을 구분합니다.
for path in (BASE_DIR, BASE_DIR / "fine-tune"):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))
from modules.notebook_workflow import resolve_settings, save_settings, plan_matrix
from modules.finetune_protocol import default_profiles
from modules.selected_campaign import default_plan, paper_baseline_plan
CAMPAIGN = "finetune_wd_zero_control"  # finetune_wd_zero_control=WD=0 Parent 재사용 + SurvFace MagFace fine-tune WD=0 대조 / parent_control=새 Parent 진단(ft WD=0.0001) / paper_baseline=SurvFace 5,319명 논문 기준선 / selected=기존 4,255명 보정분리 연구조건 / ucface_review=재현 검토 / baseline=과거 resize 캠페인
saved = resolve_settings(BASE_DIR, SETTINGS_PATH, require_saved=True)

SEEDS = list(saved["project"]["seed_list"])  # 중복 없는 정수 목록(0 <= seed < 2**32); 일부 seed만 선택 가능하며 후속 단계도 동일하게 맞춥니다.
HEADS = list(saved["campaign"]["heads"])  # "arcface", "adaface", "magface", "cosface" 중 중복 없이 선택; 02 부모는 ArcFace로 고정입니다.
SURVFACE_PAPER_RECIPE = "full_fr_resize_only"  # paper_baseline 전용: "full_fr_resize_only"(논문 Table I 기본) 또는 "full_fr_pfld_five_point_alignment"(정렬 비교 대조군)
SURVFACE_HEADS = ["magface"]  # paper_baseline 전용: 단계별 확인 시 ["arcface"] 등으로 축소 가능

CONTROL_ROOT = MODEL_ROOT / "facex/diagnostics/parent_weight_decay_zero_v1"
if CAMPAIGN == "finetune_wd_zero_control":
    # 기존 완료된 WD=0 Parent를 재사용하고 SurvFace 5,319명 MagFace의 fine-tune training.weight_decay만 0.0001 -> 0.0으로 대조합니다.
    SELECTED_ROOT = CONTROL_ROOT / "finetune_wd_zero_v1"
    SELECTION_FILE = SELECTED_ROOT / "selection.json"
    EXPERIMENT_PURPOSE = "diagnostic_control"
elif CAMPAIGN == "parent_control":
    # 02a에서 40 epoch 완료 Parent를 검증해 발행한 선택 파일만 사용합니다.
    SELECTED_ROOT = CONTROL_ROOT / "followup"
    SELECTION_FILE = SELECTED_ROOT / "selection.json"
    EXPERIMENT_PURPOSE = "diagnostic_control"
elif CAMPAIGN == "paper_baseline":
    SELECTED_ROOT = MODEL_ROOT / f"facex/selected/ucface_paper_baseline_{SURVFACE_PAPER_RECIPE}_v1"
    SELECTION_FILE = SELECTED_ROOT / "plans/selection_v2_paper_magface_only.json"
    TRAINING_PLAN = paper_baseline_plan(SURVFACE_PAPER_RECIPE, SURVFACE_HEADS, SEEDS)
    EXPERIMENT_PURPOSE = "paper_baseline"  # SurvFace 5,319 IDs / 220,888 images (calibration_unseen_in_fr=False)
else:
    # selected는 기존 연구 조건(SurvFace 4,255 IDs + FR-unseen calibration 1,064 IDs)이며 UCFace 동일 재현을 뜻하지 않습니다.
    SELECTED_ROOT = MODEL_ROOT / "facex/selected/ucface_recipes_v1"
    SELECTION_FILE = SELECTED_ROOT / "plans/selection_v2_local_protocol.json"  # 03과 04에서 같은 파일. 목록 변경 시 v2 등 새 파일명.
    TRAINING_PLAN = default_plan()  # 각 데이터셋의 4개 원래 loss, seed 8972. continuous_margin 없음.
    EXPERIMENT_PURPOSE = "local_protocol_baseline"  # author-verified UCFace reproduction으로 표시하지 않습니다.

SELECTED_REFERENCES = {}  # 추가 재사용: {run_id: {"path": "완료 checkpoint 전체 경로", "sha256": "SHA-256"}}
SELECTED_RESUME_CHECKPOINTS = {}  # MagFace는 신규 학습. 중단한 ArcFace 복구 checkpoint는 보존하며 이번 선택에는 넣지 않습니다.

# 검증 완료 후 원본/전처리 이미지와 목록을 임의로 변경하지 않는다는 전제입니다.
DATA_INTEGRITY_MODE = "immutable_verified_reuse"  # 최초 전체 검증 후 JSON 기록 재사용; "content_verified_reuse"=매번 hash, "full"=매번 hash+decode.
FORCE_DATA_REVERIFY = False  # True=이번 실행에서 전체 이미지 hash+decode 재검증. 성공 후 False로 되돌리세요.
PARENT_SEED = saved["campaign"]["parent_seed"]  # 공통 부모 seed(0 <= seed < 2**32 정수); 후속 head/seed가 공유하므로 02~04에서 유지합니다.
PRETRAIN_EPOCHS = saved["campaign"]["pretrain_epochs"]  # 부모 학습 epoch 수(양의 정수, 기본 40); 재사용 부모의 완료 epoch와 일치해야 합니다.
PRETRAIN_LR = saved["campaign"]["pretrain_lr"]  # parent의 기존 LR을 별도 유지합니다. downstream 논문 LR과 구분합니다.
PARENT_CHECKPOINT = saved["campaign"]["parent_checkpoint"]  # None=02의 고정 경로; 다른 완료 부모를 쓰려면 정확한 경로와 PARENT_SHA256를 함께 지정합니다.
PARENT_SHA256 = saved["campaign"]["parent_sha256"]  # None 또는 지정한 부모 파일의 64자리 SHA-256; 경로와 함께 지정하며 학습 조건도 검증합니다.
FINETUNE_DATASETS = list(saved["campaign"]["finetune_datasets"])  # "survface", "tinyface", "scface" 중 선택합니다. LFW/RFW는 평가 전용입니다.
FINETUNE_PROFILES = saved["finetune_profiles"]  # 외부 protocol의 분할·경로·hash입니다. 분할 인물/수량/hash를 여기서 임의로 고치지 않습니다.
AUDIT_WORKERS = 4  # 로컬 release 검사 worker 수(1 이상 정수); 현재 portable 공유 protocol에서는 사용하지 않습니다.
# 원본 데이터·공유 전처리·부모·과거 실패 기록은 이 보존 옵션의 삭제 대상이 아닙니다.
KEEP_RAW_RESULTS = saved["project"]["keep_raw_results"]  # True=이번 학습의 주기별 checkpoint 보존, False=학습 중 오래된 중간 checkpoint 정리.
NUM_WORKERS = saved["training"]["num_workers"]  # DataLoader worker 수(0~12 정수); 0=현재 프로세스에서 로딩, 큰 값은 CPU/RAM을 더 사용합니다.
PREFETCH_FACTOR = saved["training"]["prefetch_factor"]  # worker당 미리 읽을 batch 수(1 이상 정수); NUM_WORKERS>0일 때 적용합니다.
PERSISTENT_WORKERS = saved["training"]["persistent_workers"]  # True=epoch 사이 worker 유지, False=다시 생성; NUM_WORKERS=0이면 적용되지 않습니다.
TORCH_THREADS = saved["training"]["torch_threads"]  # PyTorch CPU 연산 thread 수(1 이상 정수); GPU 수나 DataLoader worker 수와 다릅니다.
LOSS_SYNC_BATCHES = saved["training"].get("loss_sync_batches", 20)  # loss 집계값을 CPU로 가져오는 batch 간격(1 이상 정수); 매 batch 학습 업데이트는 유지합니다.
RECOVERY_MINUTES = saved["training"]["recovery_minutes"]  # 재개용 checkpoint 저장 간격(분, 0보다 큰 값); 짧을수록 복구 간격과 디스크 쓰기가 촘촘해집니다.
DEVICE = saved["project"]["device"]  # "cuda:0" 또는 사용 가능한 GPU 번호("cuda:1" 등); 정식 학습은 CUDA 필수이며 CPU fallback은 없습니다.
DATASET = dict(saved["dataset"])  # VGGFace2 경로·입력 설정입니다. 112×112와 pixel_mean=127.5/pixel_std=128.0 조건을 유지합니다.
BACKBONE = dict(saved["backbone"])  # 00에서 MobileFaceNet/Resnet IR-50 선택; 1024D·dropout 0.6·out_h/out_w=7 조건을 유지합니다.
# 정식 fine-tune: optimizer=adam, lr=MobileFaceNet 0.001 / ResNet-50·100 0.0001, epoches=40, milestones=[10,20,30], lr_decay_ratio=0.1 고정.
# baseline batch_size=64는 MobileFaceNet·ResNet-50/100 공통; save_freq는 1 이상 epoch, drop_last=False 유지.
TRAINING = dict(saved["training"])  # 학습/저장 설정 사전입니다. 위 고정 학습 조건과 조절 가능한 저장 간격을 구분합니다.
HEAD_PARAMETERS = saved["heads"]  # head별 margin/scale 기준값입니다. 정식 fine-tune은 이를 검증하므로 head 선택은 HEADS에서 합니다.
PARENT_HEAD = dict(saved["parent_head"])  # 부모는 ArcFace, margin_arc=0.35/margin_am=0/scale=32로 고정합니다.
# flip_probability/affine_probability: 0~1, rotation_degrees: 0 이상, scale_range=[low,high]: 0<low<=high.
# border_mode="reflect"/"constant", center="image_center" 고정; 현재 수치를 UCFace 명시값으로 간주하지 않습니다.
AUGMENTATION = dict(saved["augmentation"])  # 증강 수치 변경은 별도 학습 조건입니다. 새 namespace와 부모 학습으로 구분합니다.
ALIGNMENT = dict(saved["alignment"])  # VGGFace2=official_landmarks, 저해상도=resize_only; 보간법은 bicubic/bilinear 중 선택합니다.
PATHS = dict(saved["paths"])  # checkpoint/export/log/report/cache 경로; 00에서 namespace별로 정하고 후속 단계가 이어받습니다.
REUSE_COMPLETED = saved["campaign"]["reuse_completed"]  # True=동일 조건의 검증된 완료 결과 재사용; False=완료 경로가 있으면 중단하며 덮어쓰지 않습니다.
EXECUTE = True  # False=계획 확인, True=이 노트북의 실제 작업 실행; 아래 세부 실행 스위치도 함께 확인합니다.
STAGE = "finetune"  # 이 노트북의 고정 단계입니다. 다른 단계는 해당 번호의 노트북을 사용합니다.
RUN_SMOKE_TEST = False  # False=정식 학습, True=합성 이미지 1 step 점검; EXECUTE=True 필요, 정식 완료 모델은 생성하지 않습니다.

RESUME_CHECKPOINTS = {}  # 과거 resize 캠페인의 복구 경로를 자동 사용하지 않습니다.

# 위 변수들을 실제 모듈 설정으로 묶어 검증합니다. 아래 전달부는 직접 편집할 필요가 없습니다.
settings = resolve_settings(BASE_DIR, SETTINGS_PATH, overrides={
 "data_integrity":dict(mode=DATA_INTEGRITY_MODE,force_reverify=FORCE_DATA_REVERIFY),
 "project":dict(seed=SEEDS[0],seed_list=SEEDS,keep_raw_results=KEEP_RAW_RESULTS,device=DEVICE),
 "campaign":dict(heads=HEADS,reuse_completed=REUSE_COMPLETED,parent_seed=PARENT_SEED,
    pretrain_epochs=PRETRAIN_EPOCHS,pretrain_lr=PRETRAIN_LR,parent_checkpoint=PARENT_CHECKPOINT,parent_sha256=PARENT_SHA256,
    finetune_datasets=FINETUNE_DATASETS),
 "finetune_profiles":FINETUNE_PROFILES,"dataset":DATASET,"backbone":BACKBONE,"heads":HEAD_PARAMETERS,
 "training":{**TRAINING,"num_workers":NUM_WORKERS,"prefetch_factor":PREFETCH_FACTOR,
    "persistent_workers":PERSISTENT_WORKERS,"torch_threads":TORCH_THREADS,"recovery_minutes":RECOVERY_MINUTES,"loss_sync_batches":LOSS_SYNC_BATCHES},
 "paths":PATHS,"parent_head":PARENT_HEAD,"augmentation":AUGMENTATION,"alignment":ALIGNMENT})

RUN_PREPROCESSING = False  # False=기존 완료 전처리 사용, True=전처리 실행; EXECUTE=True와 함께 사용합니다.
RESUME_PREPROCESSING = True  # True=중단 산출물을 검증해 재개(신규 출력도 가능), False=신규 전처리만; None 전체 처리와 함께 사용합니다.
PREPROCESS_MAX_IMAGES = None  # None=선택 학습 이미지 전체, 양의 정수=부분 점검 수; 부분 결과는 정식 학습에 사용하지 않습니다.
PREPROCESS_DEVICE = "cpu"  # "cpu"만 지원합니다. 좌표 기반 정렬/resize 처리이며 검출 모델용 GPU 설정이 아닙니다.
if CAMPAIGN == "finetune_wd_zero_control":
    from modules.parent_controls import FINETUNE_WD_ZERO_SCHEMA, build_finetune_wd_zero_selection
    from modules.selected_campaign import load_selection, inspect_selection, save_selection
    selection = load_selection(SELECTION_FILE) if SELECTION_FILE.is_file() else build_finetune_wd_zero_selection(settings, CONTROL_ROOT, SELECTED_ROOT)
    ft_ctrl = selection["settings"]["campaign"].get("finetune_control", {})
    if (selection["purpose"] != "diagnostic_control"
            or not selection["settings"]["campaign"].get("parent_binding")
            or ft_ctrl.get("schema") != FINETUNE_WD_ZERO_SCHEMA
            or selection["settings"]["training"]["weight_decay"] != 0.0):
        raise ValueError("WD=0 Parent 기반 fine-tune weight_decay=0.0 대조 선택 파일이 필요합니다.")
    settings = selection["settings"]
    TRAINING = dict(settings["training"])
    PARENT_CHECKPOINT = settings["campaign"]["parent_checkpoint"]
    PARENT_SHA256 = settings["campaign"]["parent_sha256"]
    RESUME_CHECKPOINTS = SELECTED_RESUME_CHECKPOINTS
    if RUN_SMOKE_TEST:
        raise ValueError("finetune_wd_zero_control 대조 선택은 정식 학습 전용입니다.")
elif CAMPAIGN == "parent_control":
    from modules.selected_campaign import load_selection, inspect_selection, save_selection
    selection = load_selection(SELECTION_FILE)
    if selection["purpose"] != "diagnostic_control" or not selection["settings"]["campaign"].get("parent_binding"):
        raise ValueError("02a의 새 Parent 선택 파일이 필요합니다.")
    settings = selection["settings"]
    PARENT_CHECKPOINT = settings["campaign"]["parent_checkpoint"]
    PARENT_SHA256 = settings["campaign"]["parent_sha256"]
    RESUME_CHECKPOINTS = SELECTED_RESUME_CHECKPOINTS
    if RUN_SMOKE_TEST:
        raise ValueError("parent_control 후속 선택은 정식 학습 전용입니다.")
elif CAMPAIGN in ("selected", "ucface_review", "paper_baseline"):
    from modules.selected_campaign import (make_selection, default_references, inspect_selection, save_selection, ALIGNMENT_CONTROL)
    selection = make_selection(settings, TRAINING_PLAN, SELECTED_ROOT,
        MODEL_ROOT / "facex/diagnostics" / ALIGNMENT_CONTROL, reuse_completed=REUSE_COMPLETED, purpose=EXPERIMENT_PURPOSE)
    selection["references"] = {**default_references(selection), **SELECTED_REFERENCES}
    RESUME_CHECKPOINTS = SELECTED_RESUME_CHECKPOINTS
    if RUN_SMOKE_TEST:
        raise ValueError("selected/paper_baseline은 정식 학습/재사용 경로입니다. smoke는 baseline에서 별도로 실행하세요.")
elif CAMPAIGN != "baseline":
    raise ValueError(f"Unknown campaign: {CAMPAIGN}")

from modules.ucface_reference import reproduction_review, require_reproduction_ready


## 2. 계획

In [2]:
# cell 2 : 학습 계획 및 UCFace 재현 조건 확인
# 목적: 데이터셋 × head × seed의 fine-tune 대상 run과 실행 여부를 확인합니다.

SELECTED_CAMPAIGNS = ("selected", "ucface_review", "paper_baseline", "parent_control", "finetune_wd_zero_control")
plan = inspect_selection(selection, RESUME_CHECKPOINTS, force_reverify=FORCE_DATA_REVERIFY) if CAMPAIGN in SELECTED_CAMPAIGNS else plan_matrix(settings, STAGE)
for row in plan:
    print(row)
print("예정:", len(plan), "EXECUTE:", EXECUTE, "SMOKE:", RUN_SMOKE_TEST)
if CAMPAIGN == "finetune_wd_zero_control":
    print("단일 변수 대조:", selection["settings"]["campaign"]["finetune_control"])
if CAMPAIGN in SELECTED_CAMPAIGNS:
    print("UCFace 재현 검토:", reproduction_review(selection))


{'run_id': 'survface_magface_full_fr_resize_only_s8972', 'dataset': 'survface', 'head': 'magface', 'recipe': 'full_fr_resize_only', 'seed': 8972, 'status': 'train_new', 'checkpoint': 'F:\\models\\facex\\diagnostics\\parent_weight_decay_zero_v1\\finetune_wd_zero_v1\\checkpoint\\survface_magface_full_fr_resize_only_s8972\\finetune_survface_magface_seed8972\\finetune_mobilefacenet_1024d_magface_epoch_40.pt', 'target': 'F:\\models\\facex\\diagnostics\\parent_weight_decay_zero_v1\\finetune_wd_zero_v1\\checkpoint\\survface_magface_full_fr_resize_only_s8972\\finetune_survface_magface_seed8972\\finetune_mobilefacenet_1024d_magface_epoch_40.pt'}
예정: 1 EXECUTE: True SMOKE: False
단일 변수 대조: {'schema': 'facex-finetune-weight-decay-control-v1', 'purpose': 'diagnostic_control', 'changed_parameter': 'training.weight_decay', 'reference_value': 0.0001, 'control_value': 0.0, 'parent_sha256': '640ff9bf64d22462464d65cb42acb5cc86f3549c88ec568158169015d54a7213', 'reference_selection': 'F:\\models\\facex\\dia

## 3. 고정 분할 검증과 선택적 전처리

공유 protocol을 검증하고 설정을 저장합니다. 전처리를 켜면 PREPROCESS_MAX_IMAGES로 부분 점검과 전체 준비를 구분합니다.


In [3]:
# cell 3 : 분할·전처리 준비 및 선택 파일 저장
if EXECUTE and CAMPAIGN == "ucface_review":
    require_reproduction_ready(selection)

# 목적: 실행 시 고정 분할을 검증하고 설정을 저장하며, 필요한 경우 SurvFace 5,319명 프로토콜/전처리를 준비합니다.

from modules.finetune_protocol import prepare_selected_protocols
from modules.process_runner import launch_alignment
from modules.survface_paper import prepare_survface_paper_protocol, prepare_survface_paper_inputs
print("Parent checkpoint:", PARENT_CHECKPOINT or "02 default path")
if EXECUTE and CAMPAIGN in ("selected", "paper_baseline", "parent_control", "finetune_wd_zero_control"):
    if CAMPAIGN == "paper_baseline":
        paper_proto = prepare_survface_paper_protocol(DATASET_ROOT)
        print("SurvFace 5,319명 논문 프로토콜:", paper_proto["policy"], paper_proto["num_classes"], paper_proto["images"])
        if RUN_PREPROCESSING or not Path(selection["settings"]["finetune_profiles"]["survface"]["alignment_report"]).exists():
            prep = prepare_survface_paper_inputs(BASE_DIR, DATASET_ROOT, recipe=SURVFACE_PAPER_RECIPE, max_images=PREPROCESS_MAX_IMAGES)
            print("SurvFace 5,319명 전처리 준비:", prep)
    save_selection(SELECTION_FILE, selection)
    print("04에서 사용할 선택 파일:", SELECTION_FILE)
elif EXECUTE:
    if CAMPAIGN == "baseline":
        for protocol in prepare_selected_protocols(settings, audit_workers=AUDIT_WORKERS):
            print(protocol)
    save_settings(SETTINGS_PATH, settings)
    if RUN_PREPROCESSING:
        for dataset in FINETUNE_DATASETS:
            launch_alignment(BASE_DIR, settings, dataset,
                max_images=PREPROCESS_MAX_IMAGES, device=PREPROCESS_DEVICE, resume=RESUME_PREPROCESSING)
else:
    print("Plan only: no protocol preparation, preprocessing or training.")


Parent checkpoint: F:\models\facex\diagnostics\parent_weight_decay_zero_v1\checkpoint\pretrain_arcface_seed8972\pretrain_mobilefacenet_1024d.pt
04에서 사용할 선택 파일: F:\models\facex\diagnostics\parent_weight_decay_zero_v1\finetune_wd_zero_v1\selection.json


## 4. 별도 프로세스 실행
노트북 Interrupt는 stop 파일을 만들고 학습은 optimizer step 완료 후 재개 checkpoint를 저장합니다. Kernel 강제 종료는 이 절차를 실행하지 않을 수 있으므로 출력된 stop 파일 경로 또는 progress.json을 확인하세요. MobileFaceNet 정식 fine-tune은 batch 64·Adam·40 epoch를 유지하며, Resnet batch는 00에서 지정한 값을 사용합니다.

재개 안내: `RESUME_PREPROCESSING=True`는 기존 PNG와 출처를 검증한 뒤 재사용합니다. 최초 구버전 복구는 픽셀 대조를 포함하므로 시간이 걸립니다. `recovery/`에 시도별 기록과 교체 파일을 보존합니다. 완료는 `report.json`의 `completed=true`, `failed=0` 및 `COMPLETED`로 확인합니다. 자세한 해석은 `../README.md`의 전처리 재개 절을 참고하세요.

In [4]:
# cell 4 : fine-tune 실행
if EXECUTE and CAMPAIGN == "ucface_review":
    require_reproduction_ready(selection)

# 목적: 실행 스위치와 전처리 범위가 허용할 때 별도 프로세스로 fine-tune 또는 smoke 점검을 실행합니다.

from modules.process_runner import launch_campaign
results = []
if EXECUTE and CAMPAIGN in ("selected", "paper_baseline", "parent_control", "finetune_wd_zero_control"):
    results = launch_campaign(BASE_DIR, settings, STAGE, RESUME_CHECKPOINTS,
        selection=selection, force_reverify=FORCE_DATA_REVERIFY)
elif EXECUTE and not (RUN_PREPROCESSING and PREPROCESS_MAX_IMAGES is not None):
    results = launch_campaign(BASE_DIR, settings, STAGE, RESUME_CHECKPOINTS, smoke=RUN_SMOKE_TEST)
else:
    print("계획 또는 부분 전처리 감사: 정식 학습 미실행. 전체 준비에는 PREPROCESS_MAX_IMAGES=None을 사용합니다.")


Child PID 49056; log: F:\face_datasets\.cache\facex\launches\shared_v1_boundary_mobilefacenet\4538b73932af42e3807ddd0bc51b51f2\console.log; stop request: F:\face_datasets\.cache\facex\launches\shared_v1_boundary_mobilefacenet\4538b73932af42e3807ddd0bc51b51f2\stop.request
2026-11-20 23:32:16,174 INFO Verified parent checkpoint: F:\models\facex\diagnostics\parent_weight_decay_zero_v1\checkpoint\pretrain_arcface_seed8972\pretrain_mobilefacenet_1024d.pt (SHA-256: 640ff9bf64d22462464d65cb42acb5cc86f3549c88ec568158169015d54a7213)
2026-11-20 23:32:16,174 INFO Parent checkpoint SHA-256 integrity check PASSED.
2026-11-20 23:32:16,577 INFO Verified parent checkpoint: F:\models\facex\checkpoints\shared_v1_boundary_mobilefacenet\pretrain_arcface_seed8972\pretrain_mobilefacenet_1024d.pt (SHA-256: 3ac633038bed2bf153e2fe58aae900637881aba8c78c979d22712cc995cf879a)
2026-11-20 23:32:16,577 INFO Parent checkpoint SHA-256 integrity check PASSED.
2026-11-20 23:32:18,100 INFO Verified parent checkpoint: F:\

## 5. 완료 상태

In [5]:
# cell 5 : 실행 결과 및 체크포인트 진단 요약
# 목적: fine-tune 대상 전체 대비 정식 완료 수와 각 run의 상태, 1·2·5·10·40 epoch norm/유사도 진단을 확인합니다.

from modules.survface_paper import parent_origin_comparison
for result in results:
    print(result)
if CAMPAIGN in ("selected", "ucface_review", "paper_baseline", "parent_control", "finetune_wd_zero_control") and not EXECUTE:
    print("계획 확인:", {status: sum(r["status"] == status for r in plan) for status in ("reuse_completed", "train_new", "resume", "blocked")})
    print("이번 실행에서 학습/export하지 않았습니다.")
    print("Parent vs Fine-tuned Origin 진단 요약:", parent_origin_comparison()["finding"])
else:
    print("정식 완료:", sum(r.get("completed") is True for r in results), "/", len(plan))
if CAMPAIGN == "finetune_wd_zero_control":
    from modules.parent_controls import diagnose_finetune_wd_zero_epochs
    diag = diagnose_finetune_wd_zero_epochs(selection)
    print("1단계 판정 (고정 512장 norm < 10 및 타 인물 간 코사인 유사도) - Fine-tune 직전 Parent:", diag["parent_before_finetune"])
    for ep_row in diag["epochs"]:
        print(f"Epoch {ep_row['epoch']:2d} | 기존(ft_wd=0.0001): {ep_row['reference_wd_0001']} | 신규(ft_wd=0.0): {ep_row['control_wd_0']}")
print("실패 시 summary에 미실행 범위를 기록하고 후속 작업을 중단합니다.")
print("해석 안내:", BASE_DIR / "fine-tune/README.md")


{'run_id': 'survface_magface_full_fr_resize_only_s8972', 'dataset': 'survface', 'head': 'magface', 'recipe': 'full_fr_resize_only', 'seed': 8972, 'completed': True, 'status': 'completed', 'action': 'train_new', 'checkpoint': 'F:\\models\\facex\\diagnostics\\parent_weight_decay_zero_v1\\finetune_wd_zero_v1\\checkpoint\\survface_magface_full_fr_resize_only_s8972\\finetune_survface_magface_seed8972\\finetune_mobilefacenet_1024d_magface_epoch_40.pt', 'sha256': '1dba2a7709570a646de46747aa779489da53e45f0b1cad313685db71c9f46936'}
정식 완료: 1 / 1
1단계 판정 (고정 512장 norm < 10 및 타 인물 간 코사인 유사도) - Fine-tune 직전 Parent: {'checkpoint': 'F:\\models\\facex\\diagnostics\\parent_weight_decay_zero_v1\\checkpoint\\pretrain_arcface_seed8972\\pretrain_mobilefacenet_1024d.pt', 'sha256': '640ff9bf64d22462464d65cb42acb5cc86f3549c88ec568158169015d54a7213', 'epoch': 40, 'stage': 'pretrain', 'weight_decay': 0.0, 'norm_median': 45.04, 'norm_lt_10': 0, 'diff_id_cos_mean': -0.0016, 'diff_id_cos_p95': 0.2446}
Epoch  1 | 기존